<a href="https://colab.research.google.com/github/Rohma1/AI-Assisted-Risk-of-Bias-Assessment/blob/main/risk_of_bias(module).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip -q install pymupdf sentence-transformers faiss-cpu transformers accelerate gradio pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 46.6 MB/s eta 0:00:00


In [4]:
#Check GPU
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Running on CPU")

PyTorch version: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


In [5]:
#Project structure
import os
folders = [
    "rob2_project",
    "rob2_project/papers",
    "rob2_project/data",
    "rob2_project/models",
    "rob2_project/results"
]
for folder in folders:
    os.makedirs(folder, exist_ok=True)
print("Project folders created.")

Project folders created.


In [8]:
#upload rct articles
from google.colab import files
import os

uploaded = files.upload()

for filename in uploaded.keys():
    destination = os.path.join("rob2_project", "papers", filename)

    with open(destination, "wb") as f:
        f.write(uploaded[filename])

print("\nUploaded papers:")
for filename in uploaded.keys():
    print("-", filename)


Uploaded papers:


In [9]:
#confirm no. of articles
import os
paper_dir = "rob2_project/papers"
papers = [
    f for f in os.listdir(paper_dir)
    if f.lower().endswith(".pdf")
]
print("Number of papers:", len(papers))
print()
for i, paper in enumerate(papers, 1):
    print(f"{i}. {paper}")

Number of papers: 4

1. s41598-026-56819-5.pdf
2. s41598-025-23623-6.pdf
3. s41598-026-36569-0.pdf
4. s41598-026-51855-7.pdf


In [11]:
#divide articles in page numbers
import fitz
import os

def extract_pdf_pages(pdf_path):
    document = fitz.open(pdf_path)

    pages = []

    for page_number, page in enumerate(document, start=1):
        text = page.get_text("text")

        pages.append({
            "page": page_number,
            "text": text
        })

    document.close()

    return pages

In [12]:
first_paper = os.path.join(
    "rob2_project",
    "papers",
    papers[0]
)

pages = extract_pdf_pages(first_paper)

print("Paper:", papers[0])
print("Number of pages:", len(pages))

print("\nFirst page:")
print(pages[0]["text"][:2000])

Paper: s41598-026-56819-5.pdf
Number of pages: 14

First page:
OPEN
Scientific Reports |        (2026) 16:26839 
1
| https://doi.org/10.1038/s41598-026-56819-5
Effect of a mobile-based resilience 
training program on resilience and 
well-being outcomes in individuals 
with chronic pain
Hajar Haghshenas1,2, Nilofar Pasyar3, Maryam Shaygan3 & Masoume Rambod3
Considering the limited resilience demonstrated by individuals suffering from chronic pain, this study 
aimed to explore how a mobile app designed to enhance resilience against pain affects the resilience, 
psychological and social well-being of individuals with chronic pain. In this three-arm randomized trial, 
108 individuals with chronic pain were randomly allocated to three groups. Due to the nature of the 
intervention, participant blinding was not achievable. However, both the assessor and the statistician 
remained blinded. The control group A received standard care, while the control group B participated 
in a familiarizatio

In [13]:
#Check that page numbers work
for page in pages[:5]:
    print(
        f"Page {page['page']}: "
        f"{len(page['text'])} characters"
    )


Page 1: 4532 characters
Page 2: 6301 characters
Page 3: 6554 characters
Page 4: 6845 characters
Page 5: 6880 characters


In [14]:
#Extract all(4) papers
import os
import fitz
import pandas as pd

paper_dir = "rob2_project/papers"

def extract_pdf(pdf_path):
    doc = fitz.open(pdf_path)

    pages = []

    for page_number, page in enumerate(doc, start=1):
        text = page.get_text("text").strip()

        if text:
            pages.append({
                "page": page_number,
                "text": text
            })

    doc.close()

    return pages


all_papers = {}

for filename in os.listdir(paper_dir):

    if filename.lower().endswith(".pdf"):

        path = os.path.join(paper_dir, filename)

        all_papers[filename] = extract_pdf(path)


print("Papers loaded:", len(all_papers))

for filename, pages in all_papers.items():
    print(f"{filename}: {len(pages)} pages")

Papers loaded: 4
s41598-026-56819-5.pdf: 14 pages
s41598-025-23623-6.pdf: 8 pages
s41598-026-36569-0.pdf: 16 pages
s41598-026-51855-7.pdf: 11 pages


In [15]:
#Chunking
def create_chunks(all_papers, chunk_size=1000, overlap=200):

    chunks = []

    for paper_name, pages in all_papers.items():

        for page_data in pages:

            text = page_data["text"]
            page_number = page_data["page"]

            start = 0

            while start < len(text):

                end = start + chunk_size

                chunk_text = text[start:end]

                chunks.append({
                    "paper": paper_name,
                    "page": page_number,
                    "text": chunk_text
                })

                start += chunk_size - overlap

    return chunks


chunks = create_chunks(all_papers)

print("Total chunks:", len(chunks))

print("\nExample:")
print(chunks[0])

Total chunks: 292

Example:
{'paper': 's41598-026-56819-5.pdf', 'page': 1, 'text': 'OPEN\nScientific Reports |        (2026) 16:26839 \n1\n| https://doi.org/10.1038/s41598-026-56819-5\nEffect of a mobile-based resilience \ntraining program on resilience and \nwell-being outcomes in individuals \nwith chronic pain\nHajar\xa0Haghshenas1,2, Nilofar\xa0Pasyar3\uf02a, Maryam\xa0Shaygan3 & Masoume\xa0Rambod3\nConsidering the limited resilience demonstrated by individuals suffering from chronic pain, this study \naimed to explore how a mobile app designed to enhance resilience against pain affects the resilience, \npsychological and social well-being of individuals with chronic pain. In this three-arm randomized trial, \n108 individuals with chronic pain were randomly allocated to three groups. Due to the nature of the \nintervention, participant blinding was not achievable. However, both the assessor and the statistician \nremained blinded. The control group A received standard care, while t

In [16]:
#Create embeddings
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

texts = [chunk["text"] for chunk in chunks]

embeddings = embedding_model.encode(
    texts,
    show_progress_bar=True,
    normalize_embeddings=True
)

print("Embedding shape:", embeddings.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/10 [00:00<?, ?it/s]

Embedding shape: (292, 384)


In [17]:
#FAISS index
import faiss
import numpy as np

embedding_matrix = np.array(embeddings).astype("float32")

dimension = embedding_matrix.shape[1]

index = faiss.IndexFlatIP(dimension)

index.add(embedding_matrix)

print("FAISS index created.")
print("Number of searchable chunks:", index.ntotal)

FAISS index created.
Number of searchable chunks: 292


In [20]:
#Test our evidence retrieval
def retrieve_evidence(question, paper_name, top_k=5):

    paper_chunks = [
        (i, chunk)
        for i, chunk in enumerate(chunks)
        if chunk["paper"] == paper_name
    ]

    if not paper_chunks:
        return []

    local_indices = [item[0] for item in paper_chunks]
    local_texts = [item[1]["text"] for item in paper_chunks]

    query_embedding = embedding_model.encode(
        [question],
        normalize_embeddings=True
    )

    local_embeddings = embedding_model.encode(
        local_texts,
        normalize_embeddings=True
    )

    local_embeddings = np.array(
        local_embeddings
    ).astype("float32")

    query_embedding = np.array(
        query_embedding
    ).astype("float32")

    local_index = faiss.IndexFlatIP(
        local_embeddings.shape[1]
    )

    local_index.add(local_embeddings)

    actual_k = min(top_k, len(local_texts))

    scores, indices = local_index.search(
        query_embedding,
        actual_k
    )

    results = []

    for score, local_idx in zip(scores[0], indices[0]):

        original_idx = local_indices[local_idx]

        result = chunks[original_idx].copy()
        result["score"] = float(score)

        results.append(result)

    return results

In [22]:
paper_name = "s41598-026-56819-5.pdf"

results = retrieve_evidence(
    "Was the allocation sequence random?",
    paper_name,
    top_k=5
)

for result in results:

    print("=" * 70)

    print(
        f"Paper: {result['paper']}\n"
        f"Page: {result['page']}\n"
        f"Score: {result['score']:.3f}"
    )

    print(result["text"][:1000])

Paper: s41598-026-56819-5.pdf
Page: 3
Score: 0.546
of six, two allocation slots were assigned to group A, two to group B, and two 
to group C (2A:2B:2C). This structure ensures that, once a block is completed, the numbers of participants in 
the three groups remain balanced. For a block of six positions containing two A, two B, and two C assignments, 
there was 720 possible permutations of the treatment sequence. From these, all distinct block patterns were 
enumerated and used to generate a randomization list. In total, 18 blocks (108 ÷ 6 = 18) wase used to allocate 
all participants. The final randomization schedule (sequence of blocks and the order of allocations within each 
block) were generated prior to study initiation using a computer-based random number generator. Each block 
type was assigned an identifying code, and the order of blocks were randomly determined. As participants were 
enrolled, they were receiving the next allocation in the sequence corresponding to their posi

In [23]:
#Define the assessment object
rob_assessment = {
    "study": "s41598-026-56819-5.pdf",

    "outcome": {
        "name": "Resilience",
        "timepoint": "Immediately after intervention"
    },

    "domains": {
        "D1": {
            "name": "Bias arising from the randomization process",
            "questions": [],
            "judgment": None,
            "justification": None
        },

        "D2": {
            "name": "Bias due to deviations from intended interventions",
            "questions": [],
            "judgment": None,
            "justification": None
        },

        "D3": {
            "name": "Bias due to missing outcome data",
            "questions": [],
            "judgment": None,
            "justification": None
        },

        "D4": {
            "name": "Bias in measurement of the outcome",
            "questions": [],
            "judgment": None,
            "justification": None
        },

        "D5": {
            "name": "Bias in selection of the reported result",
            "questions": [],
            "judgment": None,
            "justification": None
        }
    },

    "overall_judgment": None
}

print(rob_assessment)

{'study': 's41598-026-56819-5.pdf', 'outcome': {'name': 'Resilience', 'timepoint': 'Immediately after intervention'}, 'domains': {'D1': {'name': 'Bias arising from the randomization process', 'questions': [], 'judgment': None, 'justification': None}, 'D2': {'name': 'Bias due to deviations from intended interventions', 'questions': [], 'judgment': None, 'justification': None}, 'D3': {'name': 'Bias due to missing outcome data', 'questions': [], 'judgment': None, 'justification': None}, 'D4': {'name': 'Bias in measurement of the outcome', 'questions': [], 'judgment': None, 'justification': None}, 'D5': {'name': 'Bias in selection of the reported result', 'questions': [], 'judgment': None, 'justification': None}}, 'overall_judgment': None}


In [24]:
#Rob2-Domain 1: Randomization Process
# RoB 2 Domain 1
# Bias arising from the randomization process

domain_1 = {
    "id": "D1",
    "name": "Bias arising from the randomization process",

    "questions": {
        "D1_Q1": {
            "text": "Was the allocation sequence random?",
            "answers": [
                "Yes",
                "Probably yes",
                "Probably no",
                "No",
                "No information"
            ]
        },

        "D1_Q2": {
            "text": "Was the allocation sequence concealed until participants were enrolled and assigned to interventions?",
            "answers": [
                "Yes",
                "Probably yes",
                "Probably no",
                "No",
                "No information"
            ]
        },

        "D1_Q3": {
            "text": "Did baseline differences between intervention groups suggest a problem with the randomization process?",
            "answers": [
                "Yes",
                "Probably yes",
                "Probably no",
                "No",
                "No information"
            ]
        }
    }
}

print(domain_1["name"])

for question_id, question in domain_1["questions"].items():
    print(f"\n{question_id}")
    print(question["text"])
    print("Answers:", ", ".join(question["answers"]))

Bias arising from the randomization process

D1_Q1
Was the allocation sequence random?
Answers: Yes, Probably yes, Probably no, No, No information

D1_Q2
Was the allocation sequence concealed until participants were enrolled and assigned to interventions?
Answers: Yes, Probably yes, Probably no, No, No information

D1_Q3
Did baseline differences between intervention groups suggest a problem with the randomization process?
Answers: Yes, Probably yes, Probably no, No, No information
